# PhysIQ – Task 5: Adaptive Replanning (Multi-Turn)

Same as Tool Use Planning, but a **perturbation is injected** after the first successfully parsed action: a material changes, a support breaks, an object drifts, a tool disappears, or a new obstacle appears.

The model must **recognise the failure**, **revise its plan**, and still achieve the goal.

**Scoring:** Failure recognition (0.2) + recovery plan (0.3) + goal achieved (0.3) + efficiency (0.2)

In [ ]:
# ── Cell 1: Setup & imports ──────────────────────────────────────────────────
import sys, os, json, re
import numpy as np
import pandas as pd

sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), 'Kaggle_agi_bench'))

import kaggle_benchmarks as kbench
print('kaggle-benchmarks', kbench.__version__)

In [ ]:
# ── Cell 2: Physics engine & scenario library ────────────────────────────────
from physiq.engine import PhysIQWorld
from physiq.formats import format_as_json, format_as_ascii, format_as_nl
from physiq.generation import validate_scenario, _task_seed
from physiq.scoring import score_replan
from physiq.templates.replan import REPLAN_TEMPLATES
from physiq.templates import SCENARIO_COUNTS


def _fmt(scenario: dict, fmt: str) -> str:
    if fmt == 'ascii':
        return format_as_ascii(scenario)
    if fmt == 'nl':
        return format_as_nl(scenario)
    return format_as_json(scenario)


print(f'Replanning templates: {len(REPLAN_TEMPLATES)}')

In [ ]:
# ── Cell 3: Generate replanning scenarios ───────────────────────────────────
MASTER_SEED = 42
task_type = 'replan'
counts = SCENARIO_COUNTS[task_type]

task_seed = _task_seed(task_type, MASTER_SEED)
rng = np.random.RandomState(task_seed)

scenarios = []
for difficulty in ['easy', 'medium', 'hard']:
    target = counts[difficulty]
    collected = []
    attempt = 0
    while len(collected) < target and attempt < target * 3:
        seed = int(rng.randint(0, 2**31))
        template = REPLAN_TEMPLATES[attempt % len(REPLAN_TEMPLATES)]
        attempt += 1
        try:
            s = template.generate(difficulty, seed)
        except Exception:
            continue
        if validate_scenario(s, task_type):
            collected.append(s)
    scenarios.extend(collected)
    print(f'  {difficulty}: {len(collected)}/{target}')

print(f'Total replanning scenarios: {len(scenarios)}')

In [ ]:
# ── Cell 4: Build evaluation DataFrame ──────────────────────────────────────
rows = []
for s in scenarios:
    for fmt in ['json', 'ascii', 'nl']:
        s_with_fmt = dict(s)
        s_with_fmt['_format'] = fmt
        rows.append({
            'scenario_id':    s['id'],
            'difficulty':     s['difficulty'],
            'representation': fmt,
            'scenario_json':  json.dumps(s_with_fmt),
        })

df_replan = pd.DataFrame(rows)
print(df_replan.shape)
df_replan.head(2)

In [ ]:
# ── Cell 5: Multi-turn helpers ───────────────────────────────────────────────
MAX_TURNS = 10

_RECOGNITION_KWS = {
    'unexpected', 'changed', 'different', 'surprise', 'noticed',
    'heavier', 'broken', 'missing', 'new obstacle', 'shifted', 'moved',
    'drifted', 'disappeared', 'failed', 'did not work',
}
_RECOVERY_KWS = {
    'instead', 'alternative', 'adjust', 'change', 'modify', 'different',
    'adapt', 'replan', 'revise', 'rethink', 'new approach', 'try again',
    'new plan', 'different strategy',
}


def _recognizes_failure(response: str) -> bool:
    """True if response acknowledges the perturbation."""
    resp_lower = response.lower()
    return any(kw in resp_lower for kw in _RECOGNITION_KWS)


def _has_recovery_plan(response: str) -> bool:
    """True if response contains recovery/adaptation language."""
    resp_lower = response.lower()
    return sum(1 for kw in _RECOVERY_KWS if kw in resp_lower) >= 2


def _parse_action(response: str):
    m = re.search(r'^ACTION\s*:\s*(.+)$', response, re.IGNORECASE | re.MULTILINE)
    if not m:
        return None
    action_str = m.group(1).strip()

    pm = re.match(
        r'PLACE\s+(\w+)\s+AT\s+([\d.\-]+)\s+([\d.\-]+)(?:\s+ANGLE\s+([\d.\-]+))?',
        action_str, re.IGNORECASE
    )
    if pm:
        return {
            'type': 'place',
            'object_type': pm.group(1),
            'position': [float(pm.group(2)), float(pm.group(3))],
            'angle': float(pm.group(4)) if pm.group(4) else 0.0,
        }

    pu = re.match(
        r'PUSH\s+(\w+)\s+(?:WITH_FORCE|FORCE)\s+([\d.\-]+)\s+DIRECTION\s+([\d.\-]+)',
        action_str, re.IGNORECASE
    )
    if pu:
        return {
            'type': 'push',
            'object_id': pu.group(1),
            'force': float(pu.group(2)),
            'direction': float(pu.group(3)),
        }

    rm = re.match(r'REMOVE\s+(\w+)', action_str, re.IGNORECASE)
    if rm:
        return {'type': 'remove', 'object_id': rm.group(1)}

    return None


print('Helpers defined OK')

In [ ]:
# ── Cell 6: Task definition (multi-turn with perturbation) ──────────────────
@kbench.task(
    name='PhysIQ Adaptive Replanning',
    description=(
        'Use up to 10 actions to achieve a goal in a 2D physics world, '
        'but a surprise perturbation is injected after the first action is parsed. '
        'Tests cognitive flexibility and error recovery.'
    ),
)
def physiq_replan(llm: kbench.LLMChat, scenario_json: str) -> float:
    """Multi-turn with forced perturbation after first successfully parsed action."""
    scenario = json.loads(scenario_json)
    fmt = scenario.pop('_format', 'json')
    world = PhysIQWorld(scenario)
    goal = scenario.get('goal')
    goal_desc = scenario.get('goal_description', str(goal))
    perturbation = scenario.get('perturbation')
    tools = scenario.get('available_tools', [])
    tool_list = '\n'.join(f'  - {t}' for t in tools) if tools else '  (none specified)'

    turns_used = 0
    perturbation_injected = False
    failure_recognized = False
    recovery_plan_valid = False
    recovery_turns = 0

    prompt = (
        f"You are controlling a 2D physics simulation.\n\n"
        f"GOAL: {goal_desc}\n\n"
        f"AVAILABLE TOOLS:\n{tool_list}\n\n"
        f"CURRENT STATE:\n{_fmt(scenario, fmt)}\n\n"
        f"Available actions:\n"
        f"  PLACE <object_type> AT <x> <y> ANGLE <degrees>\n"
        f"  PUSH <object_id> FORCE <newtons> DIRECTION <degrees>\n"
        f"  REMOVE <object_id>\n\n"
        f"Explain your reasoning, then output your action on a line starting with ACTION:"
    )

    for _turn in range(MAX_TURNS):
        response = llm.prompt(prompt)
        turns_used += 1

        # After perturbation: score recognition and recovery language
        if perturbation_injected:
            if _recognizes_failure(response):
                failure_recognized = True
            if _has_recovery_plan(response):
                recovery_plan_valid = True
            recovery_turns += 1

        action = _parse_action(response)
        if action is None:
            prompt = (
                "I couldn't parse your action. Use exactly one of:\n"
                "  ACTION: PLACE <type> AT <x> <y> ANGLE <deg>\n"
                "  ACTION: PUSH <id> FORCE <n> DIRECTION <deg>\n"
                "  ACTION: REMOVE <id>\n"
                f"Turns remaining: {MAX_TURNS - turns_used}"
            )
            continue

        result_msg = world.execute_action(action)
        world.simulate(0.5)

        # Inject perturbation right after the first parsed action executes
        if not perturbation_injected and perturbation:
            world.apply_perturbation(perturbation)
            perturbation_injected = True
            perturb_desc = perturbation.get('description', 'Something unexpected happened.')
            prompt = (
                f"PERTURBATION: {perturb_desc}\n\n"
                f"UPDATED STATE:\n{_fmt(scenario, fmt)}\n\n"
                f"GOAL: {goal_desc}\n"
                f"Your previous plan may no longer work. Describe what changed, "
                f"then propose your revised action.\n"
                f"Turns remaining: {MAX_TURNS - turns_used}\n\n"
                f"ACTION: ..."
            )
            continue

        if world.check_goal(goal):
            return score_replan(
                failure_recognized=failure_recognized,
                recovery_plan_valid=recovery_plan_valid,
                goal_achieved=True,
                recovery_turns=recovery_turns,
            )

        if turns_used >= MAX_TURNS:
            break

        prompt = (
            f"Action result: {result_msg}\n\n"
            f"UPDATED STATE:\n{_fmt(scenario, fmt)}\n\n"
            f"GOAL: {goal_desc}\n"
            f"Turns remaining: {MAX_TURNS - turns_used}\n\n"
            f"Explain your reasoning, then output your action on a line starting with ACTION:"
        )

    # Goal not achieved
    return score_replan(
        failure_recognized=failure_recognized,
        recovery_plan_valid=recovery_plan_valid,
        goal_achieved=False,
        recovery_turns=recovery_turns,
    )

In [ ]:
# ── Cell 7: Dry-run sanity check ─────────────────────────────────────────────
required_cols = {'scenario_json'}
assert required_cols.issubset(df_replan.columns)
print('DataFrame columns OK:', list(df_replan.columns))
print('Rows:', len(df_replan))

s0 = json.loads(df_replan.iloc[0]['scenario_json'])
print('Sample scenario ID:', s0.get('id'))
print('Perturbation type:', s0.get('perturbation', {}).get('type', 'none'))
print('Goal:', s0.get('goal'))

In [ ]:
# ── Cell 8: Evaluation run ───────────────────────────────────────────────────
try:
    llm = kbench.llm
except AttributeError:
    print('No Kaggle LLM available (local run). Skipping evaluation.')
    llm = None

if llm is not None:
    results = []
    for _, row in df_replan.iterrows():
        run = physiq_replan.run(
            llm=llm,
            scenario_json=row['scenario_json'],
        )
        results.append({
            'scenario_id':    row['scenario_id'],
            'difficulty':     row['difficulty'],
            'representation': row['representation'],
            'score':          run.result,
        })
    df_results = pd.DataFrame(results)
    os.makedirs('../outputs', exist_ok=True)
    df_results.to_csv('../outputs/task5_replanning_results.csv', index=False)
    print('Mean score:', df_results['score'].mean())
    print(df_results.groupby(['difficulty', 'representation'])['score'].mean().unstack())

In [ ]:
# ── Cell 9: Results analysis ─────────────────────────────────────────────────
if llm is not None and 'df_results' in dir():
    import matplotlib.pyplot as plt

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    df_results.groupby('difficulty')['score'].mean().plot(
        kind='bar', ax=axes[0], color=['#4CAF50', '#FF9800', '#F44336'])
    axes[0].set_title('Replanning Score by Difficulty')
    axes[0].set_ylabel('Mean Score (0-1)')
    axes[0].set_ylim(0, 1)

    df_results.groupby('representation')['score'].mean().plot(
        kind='bar', ax=axes[1], color=['#2196F3', '#9C27B0', '#00BCD4'])
    axes[1].set_title('Replanning Score by Format')
    axes[1].set_ylabel('Mean Score (0-1)')
    axes[1].set_ylim(0, 1)

    plt.tight_layout()
    plt.savefig('../outputs/task5_replanning_results.png', dpi=150)
    plt.show()

In [ ]:
# ── Cell 10: Choose this task for the leaderboard ────────────────────────────
%choose physiq_replan